# Formative 1, Part 2 — Classical ML Classification Challenge
### Starter Notebook

**Your name(s):** _[fill in]_
**Kaggle username:** _[fill in]_
**Competition link:** _[paste the Kaggle competition URL here]_
**W&B project link:** _[paste once you've logged your first run]_

---

This notebook is a **template and a working baseline** — not a finished submission. It:
- Loads the competition data
- Explores it briefly
- Builds a simple, honest baseline (logistic regression), **fully logged to Weights & Biases**
- Generates a correctly-formatted submission file
- Gives you a reusable `log_experiment(...)` helper so every model you try afterward gets tracked the same way, with almost no extra code

**Your job:** keep this structure, replace/extend Section 5 onward with your own preprocessing decisions, additional models, and hyperparameter experiments — reusing `log_experiment(...)` for each one. Do not delete the baseline — your results table should include it as the reference point everything else is compared against.

**Remember:** you may be randomly selected to walk through your own W&B run history and explain it. Log everything, including runs that didn't work — a thin or fabricated-looking history is a problem in that session, not just for the rubric.


## 1. Introduction

_Replace this cell with 2-3 sentences: what is this competition, what are you predicting, and what metric are you optimizing? (See the competition Overview and Evaluation tabs.)_


## 2. Setup

Run this first. It installs/imports what you need and locates the data whether you're running in **Kaggle Notebooks**, **Google Colab**, or **locally**.


In [33]:
!pip install wandb
!pip install python-dotenv

In [34]:
import numpy as np
import pandas as pd
import os

pd.set_option("display.max_columns", 50)

# --- Locate the data automatically across common environments ---
CANDIDATE_DIRS = [
    "/kaggle/input",                 # Kaggle Notebooks (competition attached)
    "/content",                      # Google Colab (if you've uploaded/mounted files)
    ".",                             # local / same-folder
]

def find_data_dir():
    for base in CANDIDATE_DIRS:
        if not os.path.isdir(base):
            continue
        for root, dirs, files in os.walk(base):
            if "train.csv" in files and "test.csv" in files:
                return root
    return None

DATA_DIR = find_data_dir()
if DATA_DIR is None:
    print("Could not auto-locate train.csv/test.csv.")
    print("If you're on Colab: upload the files or mount Drive, then set DATA_DIR manually below.")
    print("If you're on Kaggle: make sure you've clicked 'Add Data' and attached this competition's dataset.")
else:
    print(f"Found data in: {DATA_DIR}")


Found data in: ./data


In [35]:
# If auto-detection above failed, set the path manually and re-run:
# DATA_DIR = "/content"  # example for Colab after uploading files

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print("train shape:", train.shape)
print("test shape:", test.shape)
train.head()


train shape: (30000, 42)
test shape: (20000, 41)


,id,num_feat_1,num_feat_2,num_feat_3,num_feat_4,num_feat_5,num_feat_6,num_feat_7,num_feat_8,num_feat_9,num_feat_10,num_feat_11,num_feat_12,num_feat_13,num_feat_14,num_feat_15,num_feat_16,num_feat_17,num_feat_18,num_feat_19,num_feat_20,num_feat_21,num_feat_22,num_feat_23,num_feat_24,num_feat_25,num_feat_26,num_feat_27,num_feat_28,num_feat_29,num_feat_30,num_feat_31,num_feat_32,num_feat_33,num_feat_34,num_feat_35,num_feat_36,num_feat_37,target,cat_region,cat_channel,cat_tier
0,row_018856,53.626073,219.076922,-1.810418,-0.159420,-1.322927,-1.615779,-1.138688,-1.615779,440.480154,-0.462100,0.108772,-5.936963,-0.185195,-1.881949,0.246464,0.826256,-1.057580,0.108772,-1.318475,-2.235114,-0.959830,-1.156057,-0.059925,-0.245132,-0.116658,-1.414168,-0.346913,0.992204,-0.040909,-0.580007,-0.924984,2.022967,-1.448010,-5.936963,0.604118,-1.753739,3.513546,0,NaN,A,mid
1,row_006719,49.359204,265.360142,4.785189,0.716050,-3.205241,0.429366,0.931481,0.429366,584.716366,0.924627,0.188854,2.066225,-1.300119,1.214380,1.651118,-1.342908,-1.792503,0.188854,1.004346,0.343773,-1.821677,0.151268,1.304049,2.911428,-1.109803,-0.002200,-2.205452,0.602929,0.179281,-0.837898,-1.646054,1.401412,-0.461406,2.066225,-0.057929,0.284302,-0.448172,0,west,A,premium
2,row_019939,45.478030,269.635184,-0.459811,-0.833381,-10.733651,-1.419081,0.840414,-1.419081,410.088750,-0.595566,0.793924,2.803522,1.116623,0.055337,0.789404,-0.736747,-0.761520,0.793924,-1.124257,3.487690,-1.953597,-0.345494,-2.686015,0.111186,-0.693300,0.137143,-2.082967,0.834090,0.576379,0.279864,-0.248444,-0.712586,0.158244,2.803522,-0.750182,-0.219530,-1.784669,0,north,B,high
3,row_049700,41.382625,248.859070,-2.269038,-0.435875,-10.932748,-3.039297,2.309878,-3.039297,NaN,1.280264,0.720175,-1.701442,-0.169711,2.216673,-0.154421,0.622095,-0.866985,0.720175,0.100486,0.444702,1.756955,-0.990839,-2.451822,-0.596345,1.283076,-2.010829,0.285625,-1.351706,3.979984,-1.618192,-0.019561,1.340699,0.976403,-1.701442,-1.242906,-0.917480,3.154757,1,north,B,mid
4,row_034441,48.776219,275.072954,-0.612385,-0.862517,5.776743,0.573600,-0.206202,0.573600,412.232672,2.227842,-0.261678,-2.724663,-0.329211,-4.051718,-1.868820,1.437544,2.258611,-0.261678,-0.166389,1.789425,-0.516547,-0.980686,2.112079,0.249812,-0.972985,0.706888,-4.977925,1.572550,0.124265,1.166693,2.527783,1.386738,0.657527,-2.724663,0.848977,0.978382,2.601208,0,north,C,high


## 3. Data Understanding

_Replace/extend this section with your own exploration. The baseline below does the bare minimum (shape, class balance, missingness) — you're expected to go further: distributions, correlations, a PCA or similar visualization, and a written interpretation of what you find. Remember: individual feature correlations with the target may look weak even where real signal exists — don't conclude "no signal" too quickly._


In [36]:
target_col = "target"
feature_cols = [c for c in train.columns if c not in ("id", target_col)]
num_cols = [c for c in feature_cols if c.startswith("num_feat")]
cat_cols = [c for c in feature_cols if c.startswith("cat_")]

print(f"{len(num_cols)} numeric features, {len(cat_cols)} categorical features")
print("\nClass balance:")
print(train[target_col].value_counts(normalize=True).round(3))

print("\nMissingness (columns with any missing values):")
miss = train.isnull().mean()
print(miss[miss > 0].sort_values(ascending=False).round(3))


37 numeric features, 3 categorical features

Class balance:
target
0    0.738
1    0.262
Name: proportion, dtype: float64

Missingness (columns with any missing values):
num_feat_7    0.119
num_feat_2    0.110
num_feat_4    0.082
num_feat_9    0.065
cat_region    0.050
dtype: float64


## 4. Experiment Tracking Setup (Weights & Biases)

Set this up **before** training anything, so your baseline run below is logged like every run after it.

Store your API key as a secret — never hardcode it in this notebook:
- **Google Colab:** left sidebar → key icon → add secret named `WANDB_API_KEY`
- **Kaggle Notebooks:** Add-ons menu → Secrets → add `WANDB_API_KEY`

This cell is written defensively: if W&B isn't set up yet, the rest of the notebook still runs — you just won't get tracking until you fix it. Don't leave it broken for long; ungraded/unlogged runs don't count.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv

load_dotenv()

WANDB_ENABLED = True
WANDB_PROJECT = "formative1-part2-BodeMurairi"  # <-- change this

try:
    import wandb
    logged_in = True

    # --- Uncomment the block matching your environment, then set logged_in = True ---

    # Google Colab:
    # from google.colab import userdata
    wandb.login(key=os.getenv("WANDB_API_KEY"))
    logged_in = True

    # Kaggle Notebooks:
    # from kaggle_secrets import UserSecretsClient
    # wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    # logged_in = True

    WANDB_ENABLED = logged_in
    if WANDB_ENABLED:
        print("W&B ready. Runs will be logged to project:", WANDB_PROJECT)
    else:
        print("W&B installed but not logged in yet — uncomment the block above for your environment.")
        print("Until then, the notebook still runs, but NOTHING is being tracked.")
except Exception as e:
    print("W&B not available — runs will NOT be logged until you fix this.")
    print("Reason:", e)


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/bode-murairi/.netrc


W&B ready. Runs will be logged to project: formative1-part2-yourname


### Reusable experiment logger

Every model you train from here on — baseline or otherwise — should go through this function. It fits your pipeline, evaluates it (holdout + cross-validation ROC-AUC), logs everything to W&B (config, metrics, confusion matrix), and keeps a local record you'll use to build your results table in Section 8.

You should not need to touch this function. Call it once per experiment with a descriptive `run_name` and a `config` dict describing what's different about that run.


In [38]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, confusion_matrix
import matplotlib.pyplot as plt

RANDOM_STATE = 42
results_log = []  # every experiment's summary lands here -> becomes your results table in Section 8

def log_experiment(run_name, pipeline, config, X_train, y_train, X_val, y_val, do_cv=True, X_full=None, y_full=None):
    '''
    Fits `pipeline`, evaluates it, logs to W&B (if enabled), and records
    a row for the results table. Returns the fitted pipeline.

    run_name : short descriptive string, e.g. "rf_depth8_lr0.05"
    config   : dict of whatever you want tracked, e.g. {"model": "RandomForest", "max_depth": 8}
    '''
    pipeline.fit(X_train, y_train)
    val_probs = pipeline.predict_proba(X_val)[:, 1]
    val_auc = roc_auc_score(y_val, val_probs)

    cv_mean, cv_std = None, None
    if do_cv and X_full is not None:
        cv_scores = cross_val_score(
            pipeline, X_full, y_full,
            cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
            scoring="roc_auc",
        )
        cv_mean, cv_std = cv_scores.mean(), cv_scores.std()

    print(f"[{run_name}] validation ROC-AUC = {val_auc:.4f}" + (f", CV ROC-AUC = {cv_mean:.4f} (+/- {cv_std:.4f})" if cv_mean else ""))

    if WANDB_ENABLED:
        try:
            run = wandb.init(project=WANDB_PROJECT, name=run_name, config=config, reinit=True, dir=DATA_DIR)
            log_dict = {"val_roc_auc": val_auc}
            if cv_mean is not None:
                log_dict.update({"cv_roc_auc_mean": cv_mean, "cv_roc_auc_std": cv_std})

            val_preds_hard = (val_probs >= 0.5).astype(int)
            cm = confusion_matrix(y_val, val_preds_hard)
            fig, ax = plt.subplots(figsize=(4, 4))
            ax.imshow(cm, cmap="Blues")
            for i in range(2):
                for j in range(2):
                    ax.text(j, i, str(cm[i, j]), ha="center", va="center")
            ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(run_name)
            wandb.log({**log_dict, "confusion_matrix": wandb.Image(fig)})
            plt.close(fig)
            run.finish()
        except Exception as e:
            print(f"  (W&B logging failed for this run: {e} — local results still recorded below)")

    results_log.append({
        "run_name": run_name,
        **config,
        "val_roc_auc": round(val_auc, 4),
        "cv_roc_auc_mean": round(cv_mean, 4) if cv_mean else None,
    })
    return pipeline


## 5. Preprocessing & Baseline Model

This section builds a **logistic regression baseline** end-to-end: impute missing values, scale numeric features, one-hot encode categoricals, then train and log it using `log_experiment(...)` from Section 4.

**This baseline is intentionally simple.** It is not meant to score well — it exists so you have:
1. A working, fully-tracked pipeline you can confirm end-to-end before building anything fancier.
2. A concrete number to beat, already sitting in `results_log` and on your W&B dashboard. If your "improved" model doesn't beat this, something is likely wrong, not just under-tuned.

Do not treat this as your final model. You are required to try at least one other model family (see the assignment instructions) and tune it properly — using the same `log_experiment(...)` pattern.


In [39]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])

X = train[feature_cols]
y = train[target_col]
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

baseline_pipeline = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
])


In [40]:

baseline_pipeline = log_experiment(
    run_name="baseline-logreg",
    pipeline=baseline_pipeline,
    config={"model": "LogisticRegression", "max_iter": 1000},
    X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val,
    X_full=X, y_full=y,
)


[baseline-logreg] validation ROC-AUC = 0.6567, CV ROC-AUC = 0.6520 (+/- 0.0042)


cv_roc_auc_mean,▁
cv_roc_auc_std,▁
val_roc_auc,▁
cv_roc_auc_mean,0.65198
cv_roc_auc_std,0.0042
val_roc_auc,0.65672


## 6. Your Models & Experiments

**This is where your actual work goes.** Use the same `log_experiment(...)` pattern from Section 5 — fit a pipeline, call `log_experiment(run_name=..., pipeline=..., config={...}, X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val, X_full=X, y_full=y)` — for every model and every hyperparameter variation you try.

Add at least one classical model family beyond the logistic regression baseline (e.g., decision tree, random forest, SVM, XGBoost, LightGBM), justify your choice in Markdown, and log at least 3 hyperparameter variations per model as required by the assignment.


### In-depth dataset understanding and data preprocessing

In [41]:
train.head()

,id,num_feat_1,num_feat_2,num_feat_3,num_feat_4,num_feat_5,num_feat_6,num_feat_7,num_feat_8,num_feat_9,num_feat_10,num_feat_11,num_feat_12,num_feat_13,num_feat_14,num_feat_15,num_feat_16,num_feat_17,num_feat_18,num_feat_19,num_feat_20,num_feat_21,num_feat_22,num_feat_23,num_feat_24,num_feat_25,num_feat_26,num_feat_27,num_feat_28,num_feat_29,num_feat_30,num_feat_31,num_feat_32,num_feat_33,num_feat_34,num_feat_35,num_feat_36,num_feat_37,target,cat_region,cat_channel,cat_tier
0,row_018856,53.626073,219.076922,-1.810418,-0.159420,-1.322927,-1.615779,-1.138688,-1.615779,440.480154,-0.462100,0.108772,-5.936963,-0.185195,-1.881949,0.246464,0.826256,-1.057580,0.108772,-1.318475,-2.235114,-0.959830,-1.156057,-0.059925,-0.245132,-0.116658,-1.414168,-0.346913,0.992204,-0.040909,-0.580007,-0.924984,2.022967,-1.448010,-5.936963,0.604118,-1.753739,3.513546,0,NaN,A,mid
1,row_006719,49.359204,265.360142,4.785189,0.716050,-3.205241,0.429366,0.931481,0.429366,584.716366,0.924627,0.188854,2.066225,-1.300119,1.214380,1.651118,-1.342908,-1.792503,0.188854,1.004346,0.343773,-1.821677,0.151268,1.304049,2.911428,-1.109803,-0.002200,-2.205452,0.602929,0.179281,-0.837898,-1.646054,1.401412,-0.461406,2.066225,-0.057929,0.284302,-0.448172,0,west,A,premium
2,row_019939,45.478030,269.635184,-0.459811,-0.833381,-10.733651,-1.419081,0.840414,-1.419081,410.088750,-0.595566,0.793924,2.803522,1.116623,0.055337,0.789404,-0.736747,-0.761520,0.793924,-1.124257,3.487690,-1.953597,-0.345494,-2.686015,0.111186,-0.693300,0.137143,-2.082967,0.834090,0.576379,0.279864,-0.248444,-0.712586,0.158244,2.803522,-0.750182,-0.219530,-1.784669,0,north,B,high
3,row_049700,41.382625,248.859070,-2.269038,-0.435875,-10.932748,-3.039297,2.309878,-3.039297,NaN,1.280264,0.720175,-1.701442,-0.169711,2.216673,-0.154421,0.622095,-0.866985,0.720175,0.100486,0.444702,1.756955,-0.990839,-2.451822,-0.596345,1.283076,-2.010829,0.285625,-1.351706,3.979984,-1.618192,-0.019561,1.340699,0.976403,-1.701442,-1.242906,-0.917480,3.154757,1,north,B,mid
4,row_034441,48.776219,275.072954,-0.612385,-0.862517,5.776743,0.573600,-0.206202,0.573600,412.232672,2.227842,-0.261678,-2.724663,-0.329211,-4.051718,-1.868820,1.437544,2.258611,-0.261678,-0.166389,1.789425,-0.516547,-0.980686,2.112079,0.249812,-0.972985,0.706888,-4.977925,1.572550,0.124265,1.166693,2.527783,1.386738,0.657527,-2.724663,0.848977,0.978382,2.601208,0,north,C,high


In [42]:
# Check for NAN
miss = train.isnull().mean()
print(miss[miss > 0].sort_values(ascending=False).round(3))

num_feat_7    0.119
num_feat_2    0.110
num_feat_4    0.082
num_feat_9    0.065
cat_region    0.050
dtype: float64


In [43]:
# Check correlation with target before preprocessing
train[num_cols].corr()

,num_feat_1,num_feat_2,num_feat_3,num_feat_4,num_feat_5,num_feat_6,num_feat_7,num_feat_8,num_feat_9,num_feat_10,num_feat_11,num_feat_12,num_feat_13,num_feat_14,num_feat_15,num_feat_16,num_feat_17,num_feat_18,num_feat_19,num_feat_20,num_feat_21,num_feat_22,num_feat_23,num_feat_24,num_feat_25,num_feat_26,num_feat_27,num_feat_28,num_feat_29,num_feat_30,num_feat_31,num_feat_32,num_feat_33,num_feat_34,num_feat_35,num_feat_36,num_feat_37
num_feat_1,1.000000,-0.316257,-0.030835,0.006105,0.431282,0.298742,-0.005869,0.298742,-0.002603,-0.008445,-0.183163,-0.226747,0.002347,-0.176355,0.488310,0.126582,0.002813,-0.183163,-0.001977,-0.213485,-0.004281,0.136208,0.002016,-0.089718,0.014813,-0.533979,0.319821,0.003774,-0.365706,0.258024,-0.232181,-0.264752,-0.002065,-0.226747,0.002997,-0.002075,-0.072239
num_feat_2,-0.316257,1.000000,0.176265,0.014808,0.030013,0.462816,0.005115,0.462816,-0.002085,0.004884,0.151860,0.321479,-0.013589,0.431357,-0.287749,0.156558,0.005016,0.151860,0.001175,0.400003,-0.005497,0.267231,-0.009142,-0.299723,-0.003329,0.225717,-0.141171,-0.000320,-0.039413,-0.134841,-0.267259,0.493962,-0.000177,0.321479,0.002834,0.005329,0.257057
num_feat_3,-0.030835,0.176265,1.000000,0.001739,0.257316,0.212633,0.004699,0.212633,-0.007409,-0.002443,-0.234800,0.258621,-0.002313,0.330116,0.219874,0.219828,0.006612,-0.234800,0.004559,-0.092479,-0.000746,-0.300549,0.008834,0.399419,0.009410,0.346817,-0.087801,0.009262,-0.088531,0.427150,0.354127,0.281923,0.004491,0.258621,-0.008713,-0.002955,-0.119659
num_feat_4,0.006105,0.014808,0.001739,1.000000,0.017502,0.008346,-0.001012,0.008346,0.005850,0.006190,-0.001168,0.005484,0.000258,0.001047,-0.008568,0.013657,-0.003049,-0.001168,-0.003028,0.003587,-0.002677,0.013994,-0.001919,-0.005156,-0.007364,-0.006303,-0.006933,-0.003293,-0.002626,0.008663,-0.004183,0.014311,0.005828,0.005484,0.004294,-0.003737,0.020581
num_feat_5,0.431282,0.030013,0.257316,0.017502,1.000000,0.359636,-0.002473,0.359636,0.000227,-0.002994,-0.440174,0.251362,-0.001483,0.062598,-0.162291,0.387253,-0.001778,-0.440174,0.014984,-0.302200,-0.002830,0.428518,-0.000083,0.134902,-0.003312,-0.053662,-0.064029,0.001326,0.104010,0.496352,0.289838,-0.066072,0.003414,0.251362,-0.000447,0.006665,0.233951
num_feat_6,0.298742,0.462816,0.212633,0.008346,0.359636,1.000000,-0.001637,1.000000,-0.006519,-0.001382,0.046330,-0.118663,-0.006846,0.038365,-0.052904,0.040578,0.014160,0.046330,0.004065,0.039140,0.000337,0.024689,0.001052,-0.016830,-0.001576,0.071231,0.174077,-0.004617,-0.099376,-0.067347,-0.064111,0.059459,0.002788,-0.118663,0.005410,0.005247,-0.057175
num_feat_7,-0.005869,0.005115,0.004699,-0.001012,-0.002473,-0.001637,1.000000,-0.001637,0.005392,0.004812,0.001803,-0.000179,0.008218,0.007978,-0.004827,0.005944,0.002585,0.001803,-0.004097,-0.004126,-0.000284,-0.004470,0.004151,-0.002770,-0.001745,0.004720,0.003183,-0.000920,-0.004254,-0.002250,0.002191,0.003822,-0.003896,-0.000179,0.006260,0.014298,0.003654
num_feat_8,0.298742,0.462816,0.212633,0.008346,0.359636,1.000000,-0.001637,1.000000,-0.006519,-0.001382,0.046330,-0.118663,-0.006846,0.038365,-0.052904,0.040578,0.014160,0.046330,0.004065,0.039140,0.000337,0.024689,0.001052,-0.016830,-0.001576,0.071231,0.174077,-0.004617,-0.099376,-0.067347,-0.064111,0.059459,0.002788,-0.118663,0.005410,0.005247,-0.057175
num_feat_9,-0.002603,-0.002085,-0.007409,0.005850,0.000227,-0.006519,0.005392,-0.006519,1.000000,0.003503,0.009875,0.003435,0.012619,0.003875,0.001211,-0.000033,0.001324,0.009875,-0.002009,0.001607,-0.002022,0.008004,0.008865,0.000394,-0.007508,-0.007468,-0.000428,0.003068,0.014975,0.003002,0.005291,-0.002592,-0.008723,0.003435,-0.002728,0.007785,0.008159
num_feat_10,-0.008445,0.004884,-0.002443,0.006190,-0.002994,-0.001382,0.004812,-0.001382,0.003503,1.000000,0.000443,0.001014,-0.001498,-0.002272,0.001232,0.000638,-0.012875,0.000443,0.007817,0.004294,-0.009472,0.007752,0.002414,0.000739,-0.009182,0.006206,-0.014213,-0.000081,0.007569,-0.008947,-0.003882,0.009241,-0.006256,0.00

### handle missing valuess

In [44]:
columns_list = train.columns
features_select = [feat for feat in columns_list if feat not in ("num_feat_8", "num_feat_18", "num_feat_34", "id", "target")]
num_features = [feat for feat in features_select if feat.startswith("num_feat")]
cat_features = [feat for feat in features_select if feat.startswith("cat_")]

In [45]:
from sklearn.linear_model import LogisticRegression
RANDOM_STATE = 42
preprocess_exp1 = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), num_features),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_features),
])

exp1_pipeline = Pipeline([
    ("prep", preprocess_exp1),
    ("clf", LogisticRegression(
    max_iter=2000,
    penalty="elasticnet",
    solver="saga",
    l1_ratio=0.5,
    class_weight=None,
    random_state=RANDOM_STATE,
)
),
])

exp1_pipeline = log_experiment(
    run_name="exp1-median-mostfrequent-impute-logistic regression",
    pipeline=exp1_pipeline,
    config={"model": "LogisticRegression", "num_impute": "median", "cat_impute": "most_frequent", "dropped_duplicates": True},
    X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val,
    X_full=X, y_full=y,
)


[exp1-median-mostfrequent-impute-logistic regression] validation ROC-AUC = 0.6568, CV ROC-AUC = 0.6520 (+/- 0.0042)


cv_roc_auc_mean,▁
cv_roc_auc_std,▁
val_roc_auc,▁
cv_roc_auc_mean,0.65198
cv_roc_auc_std,0.00419
val_roc_auc,0.65678


### Attempt to find best parameters for logistic regression using RandomSearch

In [49]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, loguniform

# solver="saga" supports l1, l2, and elasticnet, so one search space covers all penalty types
search_pipeline = Pipeline([
    ("prep", preprocess_exp1),
    ("clf", LogisticRegression(solver="saga", max_iter=2000, random_state=RANDOM_STATE)),
])

param_distributions = {
    "clf__C": loguniform(1e-3, 1e2),
    "clf__penalty": ["l1", "l2", "elasticnet"],
    "clf__l1_ratio": uniform(0, 1),        # only used when penalty="elasticnet", ignored otherwise
    "clf__class_weight": [None, "balanced"],
}

random_search = RandomizedSearchCV(
    search_pipeline,
    param_distributions=param_distributions,
    n_iter=25,                              # number of random combos to try
    scoring="roc_auc",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
)

random_search.fit(X_train, y_train)

print("Best CV ROC-AUC:", random_search.best_score_)
print("Best params:", random_search.best_params_)

Fitting 5 folds for each of 25 candidates, totalling 125 fits
Best CV ROC-AUC: 0.6528097533212622
Best params: {'clf__C': np.float64(0.033205591037519584), 'clf__class_weight': 'balanced', 'clf__l1_ratio': np.float64(0.007066305219717406), 'clf__penalty': 'l1'}


In [ ]:
best_pipeline = log_experiment(
    run_name="logreg-randomsearch-best",
    pipeline=random_search.best_estimator_,
    config={"model": "LogisticRegression", "search": "RandomizedSearchCV", **random_search.best_params_},
    X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val,
    X_full=X, y_full=y,
    do_cv=False,  # random_search already did 5-fold CV internally
)


[logreg-randomsearch-best] validation ROC-AUC = 0.6580


wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


val_roc_auc,▁
val_roc_auc,0.65802


Exception ignored in: <function ResourceTracker.__del__ at 0x7fee95d88040>
Traceback (most recent call last):
  File "/home/bode-murairi/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/bode-murairi/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/bode-murairi/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x7621941a0040>
Traceback (most recent call last):
  File "/home/bode-murairi/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 82, in __del__
  File "/home/bode-murairi/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 91, in _stop
  File "/home/bode-murairi/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 116, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception igno

In [46]:
# Your models go here. Reuse the pattern from Section 5, e.g.:
#
# from sklearn.ensemble import RandomForestClassifier
#
# my_pipeline = Pipeline([
#     ("prep", preprocess),
#     ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
# ])
#
# log_experiment(
#     run_name="descriptive-run-name",
#     pipeline=my_pipeline,
#     config={"model": "RandomForest", ...},
#     X_train=X_train, y_train=y_train, X_val=X_val, y_val=y_val,
#     X_full=X, y_full=y,
# )


## 7. Generating a Submission

This cell turns **any** fitted pipeline into a correctly-formatted submission file. It uses the baseline as the example — swap in your best model once you have one.

**Format check:** `sample_submission.csv` has two columns — `id` and `target` — where `target` is a **predicted probability**, not a 0/1 label. Using `predict_proba(...)[:, 1]` (not `predict(...)`) is what makes this correct.


In [51]:
# Refit your chosen model on the FULL training set (not just the 80% split) before predicting on test
final_model = baseline_pipeline  # <-- replace with your actual best model once you have one
final_model.fit(X, y)
test_probs = final_model.predict_proba(test[feature_cols])[:, 1]

submission = pd.DataFrame({
    "id": test["id"],
    "target": test_probs,
})

submission.to_csv("submission.csv", index=False)
print(submission.head())
print(f"\nSaved submission.csv — {submission.shape[0]} rows")


           id    target
0  row_007104  0.251735
1  row_006043  0.121918
2  row_007861  0.202070
3  row_047137  0.114058
4  row_010138  0.191881

Saved submission.csv — 20000 rows


### How to submit this file to Kaggle

1. Go to the competition page → **Submit Predictions** (or the button on the Overview tab).
2. Upload `submission.csv` (download it first if you're on Colab: files panel → download; on Kaggle Notebooks it's already in your working directory).
3. Add a short description, e.g. `"baseline logistic regression"` or `"tuned random forest, run rf_n400_depth10"` — matching your W&B run name makes it easy to trace later.
4. Check your score on the **Leaderboard** tab once scoring finishes — this is your **public** score. Remember: your grade is based on the **private** leaderboard, revealed after the deadline. Don't over-optimize for the number you can see.
5. Repeat with your own models — you get up to 5 submissions/day, and can select up to 2 as your final submissions for grading.


## 8. Results Table

Auto-built from everything logged via `log_experiment(...)` above — this should already contain your baseline and every experiment you ran. Add a `public_lb_score` and `wandb_run_url` column by hand once you've submitted to Kaggle and can see your W&B run links.


In [52]:
results_df = pd.DataFrame(results_log)
results_df["public_lb_score"] = None   # fill in after submitting to Kaggle
results_df["wandb_run_url"] = None     # paste your W&B run links here
results_df


,run_name,model,max_iter,val_roc_auc,cv_roc_auc_mean,num_impute,cat_impute,dropped_duplicates,search,clf__C,clf__class_weight,clf__l1_ratio,clf__penalty,public_lb_score,wandb_run_url
0,baseline-logreg,LogisticRegression,1000.0,0.6567,0.652,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None
1,exp1-median-mostfrequent-impute-logistic regre...,LogisticRegression,NaN,0.6568,0.652,median,most_frequent,True,NaN,NaN,NaN,NaN,NaN,None,None
2,logreg-randomsearch-best,LogisticRegression,NaN,0.6580,NaN,NaN,NaN,NaN,RandomizedSearchCV,0.033206,balanced,0.007066,l1,None,None


## 9. Discussion

_Replace with your written analysis (academic prose, not just bullet fragments): which models and choices mattered most, why you think that is, what you'd try next with more time, and what your experiments taught you about this dataset. Reference specific rows from your results table and specific W&B runs._


## 10. References

_List the competition/dataset link, and any papers, articles, or documentation you cited, in a consistent format (APA or IEEE)._
